# Backtracking through a lattice

This example shows how to use `backtrack()` to propagate a beam through the inverse of a supported element or segment. This allows reconstructing an upstream beam from a downstream beam when the tracking is reversible.

The following example uses both `ParameterBeam` and `ParticleBeam`, checks a forward/backward
round trip, and shows how to detect unsupported tracking methods.

In [1]:
import torch

import cheetah

dtype = torch.float64

The lattice contains a drift, a quadrupole, and a horizontal corrector.
Each element can use its own current tracking method, here we simply use linear tracking.

In [2]:
segment = cheetah.Segment(
    elements=[
        cheetah.Drift(length=torch.tensor(0.3, dtype=dtype), tracking_method="linear"),
        cheetah.Quadrupole(
            length=torch.tensor(0.2, dtype=dtype),
            k1=torch.tensor(0.7, dtype=dtype),
            tracking_method="linear",
            dtype=dtype,
        ),
        cheetah.HorizontalCorrector(
            length=torch.tensor(0.1, dtype=dtype),
            angle=torch.tensor(2e-4, dtype=dtype),
            dtype=dtype,
        ),
    ]
)
assert segment.supports_backtracking

## Recover a parameter beam


In [3]:
beam = cheetah.ParameterBeam.from_parameters(
    energy=torch.tensor(1e8, dtype=dtype),
    mu_x=torch.tensor(3e-4, dtype=dtype),
    mu_px=torch.tensor(1e-4, dtype=dtype),
    s=torch.tensor(1.25, dtype=dtype),
    dtype=dtype,
)
outgoing = segment.track(beam)
recovered = segment.backtrack(outgoing)

for feature in ("mu", "cov", "s"):
    torch.testing.assert_close(
        getattr(recovered, feature), getattr(beam, feature), rtol=1e-12, atol=1e-14
    )
torch.testing.assert_close(outgoing.s, beam.s + segment.length)
assert torch.equal(recovered.energy, beam.energy)
assert torch.equal(recovered.total_charge, beam.total_charge)
assert recovered.species.name == beam.species.name
print(f"Maximum mean error: {(recovered.mu - beam.mu).abs().max().item():.3e}")
print(f"Recovered s: {recovered.s.item():.2f} m")

Maximum mean error: 4.066e-20
Recovered s: 1.25 m


## Backtracking with ParticleBeam

In [4]:
torch.manual_seed(0)
particle_beam = cheetah.ParticleBeam.from_parameters(
    num_particles=1_000,
    energy=beam.energy,
    mu_x=beam.mu_x,
    mu_px=beam.mu_px,
    s=beam.s,
    dtype=dtype,
)
recovered_particles = segment.backtrack(segment.track(particle_beam))

torch.testing.assert_close(
    recovered_particles.particles, particle_beam.particles, rtol=1e-12, atol=1e-14
)
torch.testing.assert_close(recovered_particles.s, particle_beam.s)
assert torch.equal(recovered_particles.energy, particle_beam.energy)
assert torch.equal(recovered_particles.particle_charges, particle_beam.particle_charges)
assert torch.equal(
    recovered_particles.survival_probabilities, particle_beam.survival_probabilities
)
assert recovered_particles.species.name == particle_beam.species.name
print(
    "Maximum particle coordinate error: "
    f"{(recovered_particles.particles - particle_beam.particles).abs().max().item():.3e}"
)

Maximum particle coordinate error: 2.168e-19


## Inspect an inverse transfer map

Supported magnets and drifts have `inverse_first_order_transfer_map(energy, species)` with shape `(..., 7, 7)`.

Diagnostics, `Segment`, and `Superimposed` support `backtrack()` without having an explicit inverse-map implementation.

Supported magnets use analytical inverse maps. Correctors undo the exit kick before drifting backwards, including the resulting position offset.
Bends use a negative-length body map with unchanged curvature and apply inverse edges in reverse order. No generic matrix inversion is needed.

For supplied nonsingular custom affine maps, `cheetah.utils.invert_affine_map()` inverts the upper `6 x 6` block and affine offset while preserving the homogeneous bottom row. Singular linear blocks raise `torch.linalg.LinAlgError`.

In [5]:
corrector = segment.elements[-1]
forward_map = corrector.first_order_transfer_map(beam.energy, beam.species)
inverse_map = corrector.inverse_first_order_transfer_map(beam.energy, beam.species)
identity = torch.eye(7, dtype=dtype)

torch.testing.assert_close(inverse_map @ forward_map, identity, rtol=1e-12, atol=1e-14)
torch.testing.assert_close(forward_map @ inverse_map, identity, rtol=1e-12, atol=1e-14)
assert torch.equal(inverse_map[-1], identity[-1])

## Check support before backtracking

Linear backtracking is implemented for `Drift`, `Quadrupole`, `Sextupole`,
`Solenoid`, `Undulator`, `HorizontalCorrector`, `VerticalCorrector`,
`CombinedCorrector`, `Dipole`, and `RBend`.

For now, second-order and drift-kick-drift backtracking raise `NotImplementedError`.

In [6]:
unsupported = cheetah.Drift(
    length=torch.tensor(0.3, dtype=dtype), tracking_method="second_order"
)
assert not unsupported.supports_backtracking

try:
    unsupported.backtrack(beam)
except NotImplementedError as error:
    print(error)
else:
    raise AssertionError("Second-order backtracking should be rejected")
assert unsupported.tracking_method == "second_order"

Drift 'unnamed_element_4' does not support backtracking with tracking_method='second_order'.
